### 01.Import Dependencies

In [37]:
import warnings
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import (StratifiedKFold, GridSearchCV)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                            f1_score, roc_auc_score, classification_report)
warnings.filterwarnings("ignore")

### 02. Load Dataset

In [ ]:
X_train = pd.read_csv('../artifacts/X_train.csv')
Y_train = pd.read_csv('../artifacts/Y_train.csv').squeeze()
X_test = pd.read_csv('../artifacts/X_test.csv')
Y_test = pd.read_csv('../artifacts/Y_test.csv').squeeze()

### 03.Model Training

#### 03.1 Define Parameters

In [ ]:
lr_param_grid = {
    'model__max_iter': [1000, 5000],
    'model__C': [0.01, 0.1, 1, 10],
}

dt_param_grid = {
    'model__max_depth': [5, 8, 10],
    'model__min_samples_leaf': [2, 5, 10],
    'model__criterion': ['gini', 'entropy'],
}

rf_param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [8, 10, 12],
    'model__min_samples_leaf': [2, 5, 10],
    'model__class_weight': ['balanced', 'balanced_subsample', None],
}

param_grids = {
    'Logistic Regression': lr_param_grid,
    'Decision Tree': dt_param_grid,
    'Random Forest': rf_param_grid,
}

#### 03.2 Define Model

In [40]:
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier()
}

#### 03.Configure K-Fold CV

In [41]:
cv = StratifiedKFold(
    n_splits=6,
    shuffle=True,
    random_state=42
)

#### 03.4 Model Training

In [42]:
grid_search_results = {}

In [43]:
for model_name, model in models.items():
    print(f"\n--- Tuning {model_name} ---")

    # Create Pipeline: SMOTE -> Model
    pipeline = ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('model', model)
    ])

    param_grid = param_grids[model_name]

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv, scoring='f1',
        verbose=1, return_train_score=False
    )
    print(f"Fitting gridSearchCV for {model_name}")

    grid_search.fit(X_train, Y_train)

    grid_search_results[model_name] = grid_search

    print(f"{model_name} gridSearchCV completed ...")
    print(f"Best parameters: {grid_search.best_params_}")
    print(f"Best CV score: {grid_search.best_score_}")


--- Tuning Logistic Regression ---
Fitting gridSearchCV for Logistic Regression
Fitting 6 folds for each of 8 candidates, totalling 48 fits


ValueError: Invalid parameter 'model_C' for estimator Pipeline(steps=[('smote', SMOTE(random_state=42)),
                ('model', LogisticRegression())]). Valid parameters are: ['memory', 'steps', 'transform_input', 'verbose'].